In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp -r /content/drive/MyDrive/DL_Project/datasets/ffpp_cropped_faces /content/

In [10]:
!cp -r /content/drive/MyDrive/DL_Project/datasets/celebv2_cropped_faces /content/

Dependencies and Imports...

In [3]:
!pip install -q timm torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 46.5 MB/s eta 0:00:00


In [4]:
import timm

import os
import random
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, roc_curve
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

Reproducibility...

In [5]:
seed = 42

random.seed(seed)
np.random.seed(seed)

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)

cuda


Configurations...

In [7]:
[x for x in timm.list_models() if 'xception' in x]

['legacy_xception',
 'xception41',
 'xception41p',
 'xception65',
 'xception65p',
 'xception71']

In [89]:
model_name = 'xception65'

image_size = 299
batch_size = 32

learning_rate = 5e-5
weight_decay = 1e-4

maximum_epochs = 20
patience = 5

Dataset Path...

In [90]:
# root = '/content/drive/MyDrive/DL_Project/datasets'

# ffpp_root = os.path.join(root, 'ffpp_cropped_faces')

# celeb_root = os.path.join(root, 'celebv2_cropped_faces')

ffpp_root = '/content/ffpp_cropped_faces'

celeb_root = '/content/celebv2_cropped_faces'

Dataset Staistics...

In [12]:
def count_images(folder):

    real = len(os.listdir(os.path.join(folder,'real')))
    fake = len(os.listdir(os.path.join(folder,'fake')))

    return real, fake

In [13]:
for split in ['train','val','test']:
    real,fake = count_images(os.path.join(ffpp_root, split))

    print('\n', split,':', 'real =', real, '|', 'fake =', fake, '|', 'total =', real+fake)


 train : real = 7500 | fake = 6869 | total = 14369

 val : real = 1250 | fake = 1130 | total = 2380

 test : real = 1250 | fake = 1138 | total = 2388


In [14]:
real,fake = count_images(celeb_root)

print('\nceleb-df :', 'real =', real, '|', 'fake =', fake, '|', 'total =', real+fake)


celeb-df : real = 2999 | fake = 2991 | total = 5990


Dataset Class...

In [144]:
class DeepfakeDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.transform = transform
        self.samples = []

        real_dir = os.path.join(root_dir, 'real')
        fake_dir = os.path.join(root_dir, 'fake')

        for img in os.listdir(real_dir):
            self.samples.append((os.path.join(real_dir,img), 0))

        for img in os.listdir(fake_dir):
            self.samples.append((os.path.join(fake_dir,img), 1))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self,idx):

        path,label = self.samples[idx]
        image = Image.open(path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label,dtype=torch.float32), path

Transforms...

In [145]:
imagenet_mean = [0.485,0.456,0.406]
imagenet_std = [0.229,0.224,0.225]

In [146]:
train_transform = transforms.Compose([transforms.Resize((image_size,image_size)),
                                      transforms.RandomHorizontalFlip(),
                                      transforms.ColorJitter(brightness=0.1, contrast=0.1),
                                      transforms.ToTensor(),
                                      transforms.Normalize(imagenet_mean, imagenet_std)])

In [147]:
test_transform = transforms.Compose([transforms.Resize((image_size,image_size)),
                                     transforms.ToTensor(),
                                     transforms.Normalize(imagenet_mean, imagenet_std)])

Dataloaders...

In [148]:
train_ds = DeepfakeDataset(os.path.join(ffpp_root, 'train'), train_transform)

val_ds = DeepfakeDataset(os.path.join(ffpp_root, 'val'), test_transform)

test_ds = DeepfakeDataset(os.path.join(ffpp_root, 'test'), test_transform)

celeb_ds = DeepfakeDataset(celeb_root, test_transform)

In [149]:
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, persistent_workers=True, pin_memory=True)

val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True)

celeb_loader = DataLoader(celeb_ds, batch_size=batch_size, shuffle=False, pin_memory=True)

In [21]:
print('\nlength of train ffpp :', len(train_ds))
print('\nlength of val ffpp :', len(val_ds))
print('\nlength of test ffpp :', len(test_ds))
print('\nlength of celeb :', len(celeb_ds))


length of train ffpp : 14369

length of val ffpp : 2380

length of test ffpp : 2388

length of celeb : 5990


In [22]:
next(iter(train_loader))[0].shape

torch.Size([32, 3, 299, 299])

Model...

In [150]:
model = timm.create_model(model_name, pretrained=True, num_classes=1)

model = model.to(device)

In [151]:
print(model.get_classifier())

Linear(in_features=2048, out_features=1, bias=True)


In [152]:
total_parameters = sum(p.numel() for p in model.parameters())

print(f'\nparameters: {total_parameters:,}')


parameters: 37,869,361


Loss / Optimizer ...

In [153]:
criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

In [154]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

AMP...

In [155]:
scaler = torch.amp.GradScaler('cuda')

Evaluation Function...

In [156]:
@torch.no_grad()

def evaluate(model, loader, criterion):

    model.eval()

    running_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []
    all_logits = []

    is_first_batch = True

    for images, labels, paths in tqdm(loader, leave=False):

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images).squeeze(1)
        labels = labels.float().view_as(outputs)

        if is_first_batch:
            print('\noutputs shape :', outputs.shape)
            print('labels shape  :', labels.shape)

            print('\noutputs dtype :', outputs.dtype)
            print('labels dtype  :', labels.dtype)

            print('\noutput range  :', outputs.min().item(), outputs.max().item())

        # safety checks..

        if torch.isnan(outputs).any():
            raise ValueError('nan detected in model outputs')

        if torch.isinf(outputs).any():
            raise ValueError('inf detected in model outputs')

        loss = criterion(outputs, labels)

        sample_losses = torch.nn.functional.binary_cross_entropy_with_logits(outputs, labels, reduction='none')
        max_sample_loss = sample_losses.max().item()

        if max_sample_loss > 100:
            print('\nmax sample loss :', max_sample_loss)

            idx = sample_losses.argmax()

            print('\npath :', paths[idx])
            print('\nlabel:', labels[idx].item())
            print('logit:', outputs[idx].item())

        if is_first_batch:
           print('\nbatch loss :', loss.item())

        running_loss += loss.item() * images.size(0)

        probs = torch.sigmoid(outputs)

        if is_first_batch:
            print('\nprob range :', probs.min().item(), probs.max().item())
            is_first_batch = False

        if torch.isnan(probs).any():
            raise ValueError('nan detected in probabilities')

        preds = (probs >= 0.5).float()

        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_logits.extend(outputs.cpu().numpy())
        print(np.min(all_logits), np.max(all_logits))

    epoch_loss = running_loss / len(loader.dataset)

    auc = roc_auc_score(all_labels, all_probs)

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    return {
        'loss': epoch_loss,
        'auc': auc,
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'labels': np.array(all_labels),
        'probs': np.array(all_probs),
        'preds': np.array(all_preds)
    }

Training Loop...

In [157]:
history = {'train_loss': [], 'val_loss': [], 'val_auc': [],
           'val_acc': [], 'val_precision': [], 'val_recall': [], 'val_f1': []}

best_auc = 0.0
patience_counter = 0

save_dir = '/content/drive/MyDrive/DL_Project/checkpoints'
os.makedirs(save_dir, exist_ok=True)

best_model_path = os.path.join(save_dir,'best_xception65.pth')

In [158]:
images, labels, paths = next(iter(train_loader))

print(labels.dtype)
print(labels.min())
print(labels.max())

torch.float32
tensor(0.)
tensor(1.)


In [159]:
for epoch in range(maximum_epochs):

    print(f'\nepoch [{epoch+1}/{maximum_epochs}]')

    # training...

    model.train()

    running_loss = 0.0

    pbar = tqdm(train_loader)

    for images, labels, paths in pbar:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float()

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):

            outputs = model(images).squeeze(1)
            labels = labels.view_as(outputs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        pbar.set_postfix(loss=f'{loss.item():.3f}')

    train_loss = running_loss/len(train_loader.dataset)

    # validation..

    val_metrics = evaluate(model, val_loader, criterion)

    val_loss = val_metrics['loss']
    val_auc = val_metrics['auc']

    val_acc = val_metrics['accuracy']
    val_precision = val_metrics['precision']
    val_recall = val_metrics['recall']
    val_f1 = val_metrics['f1']

    scheduler.step(val_auc)

    history['train_loss'].append(train_loss)

    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)

    history['val_acc'].append(val_acc)
    history['val_precision'].append(val_precision)
    history['val_recall'].append(val_recall)
    history['val_f1'].append(val_f1)

    print(f'\ntrain loss  : {train_loss:.4f}')

    print(f'\nval loss  : {val_loss:.4f}')
    print(f'\nval auc  : {val_auc:.4f}')

    print(f'\nval accuracy  : {val_acc:.4f}')
    print(f'val precision  : {val_precision:.4f}')
    print(f'val recall  : {val_recall:.4f}')
    print(f'val f1  : {val_f1:.4f}')

    # early stopping..

    if val_auc > best_auc:
        best_auc = val_auc
        patience_counter = 0

        torch.save({'epoch': epoch + 1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_auc': best_auc},best_model_path)

        print(f'\nbest model saved (auc={best_auc:.4f})')

    else:
        patience_counter += 1
        print(f'\nno improvement ({patience_counter}/{patience})')

    if patience_counter >= patience:
        print('\nearly stopping triggered...')

        break


epoch [1/20]


  0%|          | 0/450 [00:00<?, ?it/s]

  0%|          | 0/75 [00:00<?, ?it/s]


outputs shape : torch.Size([32])
labels shape  : torch.Size([32])

outputs dtype : torch.float32
labels dtype  : torch.float32

output range  : -172.1659393310547 3.7377688884735107

batch loss : 0.4732603132724762

prob range : 0.0 0.9767464399337769
-172.16594 3.737769
-172.16594 4.2598257
-172.16594 4.2598257
-172.16594 4.2598257
-185.99014 4.2598257
-185.99014 4.2598257
-339.5451 4.2598257
-811.1669 4.2598257
-811.1669 4.3437676
-811.1669 4.3437676
-811.1669 4.3437676
-811.1669 4.3437676
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.1669 4.798332
-811.166

  0%|          | 0/450 [00:00<?, ?it/s]

  0%|          | 0/75 [00:00<?, ?it/s]


outputs shape : torch.Size([32])
labels shape  : torch.Size([32])

outputs dtype : torch.float32
labels dtype  : torch.float32

output range  : -1109.6922607421875 5.351569175720215

batch loss : 0.4211696684360504

prob range : 0.0 0.9952816367149353
-1109.6923 5.351569
-1109.6923 5.4694743
-1109.6923 5.4694743
-1109.6923 5.4694743
-1109.6923 5.4694743
-1109.6923 5.4694743
-1150.2999 5.4694743
-1150.2999 5.4694743
-1150.2999 5.4694743
-1150.2999 5.4694743
-2192.2756 5.4694743
-2192.2756 5.4694743
-2192.2756 5.4694743
-2192.2756 5.4694743
-2192.2756 5.4694743
-2907.513 5.4694743
-2907.513 5.4694743
-2907.513 5.4694743
-2907.513 5.4694743
-2907.513 5.4694743
-2907.513 5.4694743
-2907.513 5.4694743
-2907.513 5.4694743
-2907.513 5.4694743
-2907.513 5.4694743
-2907.513 5.4694743
-2907.513 5.4694743
-2907.513 5.4694743
-2907.513 5.4694743
-2907.513 5.4694743
-3185.0002 5.4694743
-3185.0002 5.4694743
-3185.0002 6.518665
-3640.3513 6.518665
-3640.3513 6.518665
-3640.3513 6.518665
-3640.3513 

  0%|          | 0/450 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
history_df = pd.DataFrame(history)

results_dir = '/content/drive/MyDrive/DL_Project/results'

os.makedirs(results_dir, exist_ok=True)
history_df.to_csv(os.path.join(results_dir, 'xception65_training_history.csv'), index=False)

print('training history saved...')

Learning Curves...

FF++ Test Evaluation...

ROC Curve...

Confusion Matrix...

Celeb-DF Generalization...

Generalization Gap...

Export Results...